# Real-Time Indian Sign Language (ISL) Recognition System

**Author:** Vatsala Misra  
**Tech Stack:** Python · MediaPipe · TensorFlow · OpenCV  

## Overview
An end-to-end pipeline that recognises **36 ISL gestures** (26 alphabets A–Z + digits 0–9) in real time using a webcam.

### Architecture
1. **MediaPipe Hands** extracts 21 hand landmarks (x, y, z) → 63 features per frame
2. **Dense Neural Network** (256→128→36) with BatchNorm + Dropout classifies gestures
3. **Temporal smoothing** (sliding window of 5 frames) removes flickering for stable predictions
4. **Data augmentation** (rotation, scaling, noise on landmarks) improves robustness

### Results
- Achieves **0.97–1.00 confidence** on most gestures in live webcam testing
- Robust across varied backgrounds and lighting conditions

## 1. Setup & Imports

In [ ]:
!pip install tensorflow opencv-python mediapipe numpy matplotlib scikit-learn

import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
from collections import deque
import os
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## 2. Data Loading & Landmark Extraction

MediaPipe Hands detects 21 keypoints on the hand. Each keypoint has (x, y, z) coordinates,
giving us a **63-dimensional feature vector** per frame — much more efficient than raw pixel CNNs.

In [ ]:
# Initialize MediaPipe for static image processing
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands_static = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.7
)

def extract_landmarks(img_path):
    """Extract 21 hand landmarks (63 features) from an image using MediaPipe."""
    image = cv2.imread(img_path)
    if image is None:
        return None
    image = cv2.resize(image, (640, 480))
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands_static.process(image_rgb)

    if results.multi_hand_landmarks:
        landmarks = []
        for lm in results.multi_hand_landmarks[0].landmark:
            landmarks.extend([lm.x, lm.y, lm.z])  # 21 points × 3 = 63 values
        return np.array(landmarks)
    return None

# Load dataset — expects folder structure: ISL_Dataset/A/, ISL_Dataset/B/, ..., ISL_Dataset/9/
dataset_path = "ISL_Dataset"
class_labels = sorted([f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))])
label_map = {name: idx for idx, name in enumerate(class_labels)}

X, y = [], []
for class_name in class_labels:
    class_dir = os.path.join(dataset_path, class_name)
    for img_name in os.listdir(class_dir):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            landmarks = extract_landmarks(os.path.join(class_dir, img_name))
            if landmarks is not None:
                X.append(landmarks)
                y.append(label_map[class_name])

X = np.array(X)
y = np.array(y)

# Save extracted features for reuse
np.save("X.npy", X)
np.save("y.npy", y)
print(f"Extracted {len(X)} samples across {len(class_labels)} classes")
print(f"Feature shape: {X.shape}")

## 3. Data Augmentation

Since landmark-based models can be sensitive to hand orientation and scale,
we augment by applying **geometric transformations** directly on the 63D landmark vectors.
This avoids the need for expensive image augmentation and is computationally lightweight.

In [ ]:
def augment_landmarks(landmarks, rotation_range=15, scale_range=0.1, noise_range=0.02):
    """
    Apply geometric transformations to hand landmarks for data augmentation.
    
    Args:
        landmarks: 63-length array (21 points × x,y,z)
        rotation_range: max rotation angle in degrees
        scale_range: max scale variation (±)
        noise_range: gaussian noise std
    Returns:
        Augmented 63-length landmark array
    """
    landmarks = landmarks.reshape(21, 3)
    wrist = landmarks[0].copy()  # Use wrist as rotation anchor

    # 1. Rotation around wrist point
    angle = np.radians(np.random.uniform(-rotation_range, rotation_range))
    rot_matrix = np.array([
        [np.cos(angle), -np.sin(angle), 0],
        [np.sin(angle),  np.cos(angle), 0],
        [0,              0,             1]
    ])
    landmarks[:, :2] = np.dot(landmarks[:, :2] - wrist[:2], rot_matrix[:2, :2]) + wrist[:2]

    # 2. Random scale
    scale = np.random.uniform(1 - scale_range, 1 + scale_range)
    landmarks = wrist + (landmarks - wrist) * scale

    # 3. Small gaussian noise
    landmarks += np.random.normal(0, noise_range, landmarks.shape)

    return landmarks.flatten()

# Generate 5 augmented versions per sample (6× dataset size)
X_aug = np.array([augment_landmarks(lm) for lm in X for _ in range(5)])
y_aug = np.repeat(y, 5)

X_final = np.concatenate([X, X_aug])
y_final = np.concatenate([y, y_aug])

print(f"Original samples: {len(X)}")
print(f"After augmentation: {len(X_final)} total samples")

## 4. Model Training

We use a **fully connected network** on the 63D landmark features.
Key design decisions:
- **BatchNormalization** for training stability
- **Dropout** to prevent overfitting on the small dataset
- **Class weights** to handle class imbalance across 36 gesture classes
- **EarlyStopping + ReduceLROnPlateau** to prevent overtraining

In [ ]:
# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42, stratify=y_final
)

# Handle class imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y_final), y=y_final)
class_weight_dict = dict(enumerate(class_weights))

# Model architecture
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(63,)),                          # 63 landmark features
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(class_labels), activation='softmax')  # 36 classes (A-Z, 0-9)
], name="ISL_Classifier")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Train
history = model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_val, y_val),
    class_weight=class_weight_dict,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.2, patience=5, verbose=1)
    ]
)

# Save model
model.save("isl_model_augmented.h5")
print("Model saved as isl_model_augmented.h5")

## 5. Training Results Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 6. Real-Time ISL Detection

The inference pipeline:
1. Captures webcam frames via OpenCV
2. MediaPipe detects hand and returns 21 landmarks
3. Model predicts gesture class from 63D feature vector
4. **Temporal smoothing** (deque of 5 frames + average voting) stabilises output
5. Displays predicted sign + confidence score on frame

**Press `q` to quit.**

In [ ]:
def real_time_isl_detection(model_path="isl_model_augmented.h5"):
    """
    Run real-time ISL gesture recognition from webcam.
    Press 'q' to quit.
    """
    # Load model and class labels
    model = tf.keras.models.load_model(model_path)
    class_labels = [chr(i) for i in range(65, 91)] + [str(i) for i in range(10)]  # A-Z, 0-9

    # MediaPipe for real-time (static_image_mode=False enables tracking)
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.7,
        min_tracking_confidence=0.5
    )
    mp_drawing = mp.solutions.drawing_utils

    # Temporal smoothing: average predictions over last 5 frames
    prediction_history = deque(maxlen=5)

    cap = cv2.VideoCapture(0)
    print("Starting ISL detection... Press 'q' to quit.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)  # Mirror for intuitive interaction
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)

        if results.multi_hand_landmarks:
            # Draw landmark skeleton on frame
            mp_drawing.draw_landmarks(
                frame, results.multi_hand_landmarks[0], mp_hands.HAND_CONNECTIONS
            )

            # Extract 63 landmark features
            landmarks = []
            for lm in results.multi_hand_landmarks[0].landmark:
                landmarks.extend([lm.x, lm.y, lm.z])

            # Predict and store in history
            pred = model.predict(np.array([landmarks]), verbose=0)[0]
            prediction_history.append(pred)

            # Smooth predictions over last 5 frames
            if len(prediction_history) == 5:
                avg_pred = np.mean(prediction_history, axis=0)
                predicted_idx = np.argmax(avg_pred)
                confidence = avg_pred[predicted_idx]

                # Only display if confidence > 50%
                if confidence > 0.5:
                    cv2.putText(
                        frame,
                        f"{class_labels[predicted_idx]} ({confidence:.2f})",
                        (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2
                    )

        cv2.imshow("ISL Detection (Augmented Model)", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Detection stopped.")


# Run detection
real_time_isl_detection()